### RAG pipeline

#### Data ingestion + chunking

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
import pandas as pd

import os
from pathlib import Path

# The below code is used to access packages from the root
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..", "..")))

from app.agent.rag.ingestion.data_ingestion import process_and_load_file
from app.agent.rag.ingestion.embeddings import get_embedding_model, generate_embeddings

In [3]:
print(os.getcwd())

c:\Users\USER\Documents\AI projects\Denseless-local\app\agent


In [8]:
# Upload your pdf material to "app\pdfs" dir
# Insert its path as the parameter
docs = process_and_load_file("../pdfs/Incident Response.pdf")
print(docs)

INFO: split_pdf event=plan_created operation_id=94cb306e-042f-4896-8186-65f94f061e35 filename=Incident Response.pdf strategy=hi_res page_range=1-5 page_count=5 split_size=2 chunk_count=3 concurrency=5 allow_failed=False cache_mode=disabled timeout_seconds=None retry_config_mode=sdk_default_or_unset


PDF page count: 5
Processing: Incident Response.pdf


INFO: HTTP Request: GET https://api.unstructuredapp.io/general/docs "HTTP/1.1 200 OK"
INFO: split_pdf event=batch_start operation_id=94cb306e-042f-4896-8186-65f94f061e35 chunk_count=3 concurrency=5 allow_failed=False client_timeout_seconds=None future_timeout_seconds=3605 num_waves=1
INFO: HTTP Request: POST https://api.unstructuredapp.io/general/v0/general "HTTP/1.1 200 OK"
INFO: HTTP Request: POST https://api.unstructuredapp.io/general/v0/general "HTTP/1.1 200 OK"
INFO: HTTP Request: POST https://api.unstructuredapp.io/general/v0/general "HTTP/1.1 200 OK"
INFO: split_pdf event=batch_complete operation_id=94cb306e-042f-4896-8186-65f94f061e35 chunk_count=3 success_count=3 failure_count=0 transport_failure_count=0 elapsed_ms=8530 allow_failed=False


  → Parsed into 51 document(s)
[Document(metadata={'section': 'WEKK 5 INCIDENCE RESPONSE', 'category': 'Title', 'page_number': 1, 'element_id': 'bde6e6d043bc018c3d867dfd672fe959', 'filename': 'Incident Response.pdf'}, page_content='WEKK 5 INCIDENCE RESPONSE'), Document(metadata={'section': 'Introduction', 'category': 'NarrativeText', 'page_number': 1, 'element_id': '13366e6ff44073ab081245d6819e3980', 'filename': 'Incident Response.pdf'}, page_content='Incident response is an organized, strategic approach to detecting and managing cyber attacks in ways that minimize damage, recovery time and total costs.'), Document(metadata={'section': 'Introduction', 'category': 'NarrativeText', 'page_number': 1, 'element_id': '873d5a8aeb84783ce3cf59cafb9fcb72', 'filename': 'Incident Response.pdf'}, page_content="Strictly speaking, incident response is a subset of incident management. Incident management is an umbrella term for an enterprise's broad handling of cyber attacks, involving diverse stakeho

In [9]:
pd.set_option('display.max_rows', None)

df = pd.DataFrame([
    {**doc.metadata, "content_preview": doc.page_content[:300], "character count": len(doc.page_content)}
    for doc in docs
])
# df

In [10]:
df.head

<bound method NDFrame.head of                                               section       category  \
0                           WEKK 5 INCIDENCE RESPONSE          Title   
1                                        Introduction  NarrativeText   
2                                        Introduction  NarrativeText   
3                         Types of security incidents          Title   
4                         Types of security incidents  NarrativeText   
5                         Types of security incidents  NarrativeText   
6                         Types of security incidents  NarrativeText   
7                         Types of security incidents       ListItem   
8                         Types of security incidents       ListItem   
9                         Types of security incidents       ListItem   
10                        Types of security incidents       ListItem   
11                        Types of security incidents       ListItem   
12                        Types of

In [11]:
pd.set_option('display.max_rows', 40)
print(df)

                                       section       category  page_number  \
0                    WEKK 5 INCIDENCE RESPONSE          Title            1   
1                                 Introduction  NarrativeText            1   
2                                 Introduction  NarrativeText            1   
3                  Types of security incidents          Title            1   
4                  Types of security incidents  NarrativeText            1   
..                                         ...            ...          ...   
46  Why is an Incident Response Plan Important       ListItem            4   
47  Why is an Incident Response Plan Important       ListItem            4   
48  Why is an Incident Response Plan Important       ListItem            5   
49  Why is an Incident Response Plan Important       ListItem            5   
50  Why is an Incident Response Plan Important       ListItem            5   

                          element_id               filename  \


In [12]:
len(docs[11].page_content)

18

#### Embeddings

In [13]:
embedder = get_embedding_model()
vectors = generate_embeddings(docs, embedder)
print(vectors)

Loading embedding model: BAAI/bge-small-en-v1.5 (device=cpu)


INFO: Load pretrained SentenceTransformer: BAAI/bge-small-en-v1.5
INFO: HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
INFO: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/modules.json "HTTP/1.1 200 OK"
INFO: HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
INFO: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/config_sentence_transformers.json "HTTP/1.1 200 OK"
INFO: HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
INFO: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/confi

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
INFO: HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/config.json "HTTP/1.1 200 OK"
INFO: HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
INFO: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/tokenizer_config.json "HTTP/1.1 200 OK"
INFO: HTTP Request: GET https://huggingface.co/api/models/BAAI/bge-small-en-v1.5/tree

  → Embedding model loaded successfully. Embedding dimension: 384
Generating embeddings for 51 document chunk(s)...


Embedding documents: 100%|██████████| 2/2 [00:01<00:00,  1.56it/s]

  → Successfully generated 51 embedding vectors.
  → Embeddings shape: (51, 384)
[[-0.0228095   0.02379     0.05545533 ... -0.00483634  0.0681124
   0.02830725]
 [-0.04193354  0.06092418  0.03404565 ... -0.00474243  0.02409485
  -0.0530497 ]
 [-0.02999221  0.03288675  0.04344953 ... -0.00868725 -0.01018949
  -0.02232714]
 ...
 [-0.03641431  0.00808547  0.01744381 ...  0.02504822  0.0193478
   0.02478076]
 [-0.03033938  0.02446019  0.0029533  ... -0.04359387  0.07814716
  -0.00025909]
 [-0.0297759   0.03453562 -0.01875774 ...  0.01069011  0.03605673
  -0.0103386 ]]


#### Vector store

In [14]:
from app.agent.rag.ingestion.vector_store import get_vector_store, add_documents_to_chroma

In [15]:
# Use any number as the student id - However, be consistent with it
# every other place it is required as a parameter
# It is advised to clear the "app\chroma_db" dir's contents
# each time this notebook is run
store = get_vector_store(student_id="1001", embedder=embedder)

Vector store backend: 'chroma' | student: '1001'
Initialising Chroma store — student: '1001'
  → Collection : student_1001
  → Persist dir: ../chroma_db
  → Chroma store ready. 
Documents in collection: 51


In [8]:
# Upon successful creation of vector store, you can add the 
# document's embeddings and chunks to the db using the 
# function below by filling its parameters
add_documents_to_chroma(store, vectors, docs, False, "PQ", "Incident Response", "Incident Response")

  [vector_store] 51 existing chunk(s) found in collection.
  [vector_store] 51 duplicate chunk(s) skipped.
  [vector_store] ✓ All documents already exist in store — nothing to add.


In [ ]:
# You can add multiple docs
# add_documents_to_chroma(store, vectors, docs, False, "CSC 403", "Software Engineering Intro", "Software Engineering Intro")

  [vector_store] 49 existing chunk(s) found in collection.
  [vector_store] 49 duplicate chunk(s) skipped.
  [vector_store] ✓ All documents already exist in store — nothing to add.


### Chains

You can now interact with the chains to perform the ai's various tasks.

#### Notes chain

In [16]:
from app.agent.rag.chains.notes_chain import run_notes_chain
from app.agent.rag.retrieval.retriever import get_topic_chunks
from langchain_ollama import ChatOllama
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI

In [10]:
# Instantiate your llm (local or paid)
# local:
llm = ChatOllama(model="gemma3:1b")
# load_dotenv()
# api_key = os.environ.get('GOOGLE_API_KEY')
# llm = ChatGoogleGenerativeAI(
#     model="gemini-2.5-flash-lite",  
#     project="denseless",       
#     location="us-central1",             
#     vertexai=True,
# )

In [11]:
# Fetch all the topic's chunks for notes generation by supplying
# the parameters in the get_topic_chunks function
chunks_topic =  get_topic_chunks(store, "Incident Response", "PQ")

[Retriever] Topic sweep — fetching all chunks from store...
[Retriever] → 43 chunk(s) merged into existing sections (duplicate titles collapsed).
[Retriever] → 51 chunk(s) across 8 section(s).
[Retriever] → Topic map built: 8 section(s), all chunks retained.


In [ ]:
# Case: A fast student

# Enter student_id as student_{number used for vector store}
pdf_path = run_notes_chain(
      student_id    = "student_1019",
      topic_map     = chunks_topic,
      current_topic = "Software Engineering Intro",
      weak_topics=[],
      strong_topics = [],
      course="CSC 403",
      learning_pace = "fast",
      llm           = llm,
      embedder      = embedder,
      store         = store,
  )

  [token_service] Tokens remaining for 'student_1019': 6,832,939
  [token_guard] Checking tokens — student: student_1019 | remaining: 6832939 | chain: run_notes_chain

[notes_chain] ═══ Starting notes generation ═══
[notes_chain] Mode: Local Ollama
[notes_chain] Student: student_1019 | Topic: Software Engineering Intro | Pace: fast
[notes_chain] Sections to process: 8

[notes_chain] Processing section 1/8: 'WEKK 5 INCIDENCE RESPONSE'


INFO: HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


[notes_chain] ⚠ Section 'WEKK 5 INCIDENCE RESPONSE' returned empty condensed_content (attempt 1) — retrying.


INFO: HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


[notes_chain] ⚠ Section 'WEKK 5 INCIDENCE RESPONSE' returned empty condensed_content (attempt 2) — retrying.
[notes_chain] ✗ Section 'WEKK 5 INCIDENCE RESPONSE' failed after 2 attempts. Inserting placeholder.

[notes_chain] Processing section 2/8: 'Introduction'


...

[notes_chain] ⚠ Section 'Types of incident response teams' returned empty condensed_content (attempt 1) — retrying.


INFO: HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


[notes_chain] ✓ Section 'Types of incident response teams' condensed.

[notes_chain] Processing section 8/8: 'Why is an Incident Response Plan Important'


INFO: HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


[notes_chain] ✓ Section 'Why is an Incident Response Plan Important?' condensed.
[notes_chain] PDF built: c:\Users\USER\Documents\AI projects\Denseless-local\data\notes\student_1019_Software_Engineering_Intro_20260624_103253.pdf

[notes_chain] ═══ Notes generation complete ═══
[notes_chain] PDF: c:\Users\USER\Documents\AI projects\Denseless-local\data\notes\student_1019_Software_Engineering_Intro_20260624_103253.pdf
[notes_chain] Sections completed: 8/8
[notes_chain] Total tokens — in: 10,613 | out: 2,721 | total: 13,334
  [token_service] Deducted 13,334 tokens — student: 'student_1019' | chain: run_notes_chain | in: 10,613 | out: 2,721 | remaining: 6,819,605
  [token_guard] ✓ Tokens deducted — student: student_1019 | consumed: 13334 | remaining: 6819605


In [ ]:
# Case: An avg student

# Enter student_id as student_{number used for vector store}
pdf_path = run_notes_chain(
      student_id    = "student_1019",
      topic_map     = chunks_topic,
      current_topic = "Incident Response",
      weak_topics=[],
      strong_topics = [],
      course="PQ",
      learning_pace = "average",
      llm           = llm,
      embedder      = embedder,
      store         = store,
  )

  [token_service] Tokens remaining for 'student_1019': 6,819,605
  [token_guard] Checking tokens — student: student_1019 | remaining: 6819605 | chain: run_notes_chain

[notes_chain] ═══ Starting notes generation ═══
[notes_chain] Mode: Local Ollama
[notes_chain] Student: student_1019 | Topic: Incident Response | Pace: average
[notes_chain] Sections to process: 8

[notes_chain] Processing section 1/8: 'WEKK 5 INCIDENCE RESPONSE'


INFO: HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


[notes_chain] ✓ Section 'WEKK 5 INCIDENCE RESPONSE' condensed.

[notes_chain] Processing section 2/8: 'Introduction'


INFO: HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


[notes_chain] ⚠ Section 'Introduction' returned empty condensed_content (attempt 1) — retrying.


...

[notes_chain] ✓ Retry 2 succeeded for 'Incident response frameworks: Phases of incident r'.
[notes_chain] ✓ Section 'Incident Response Framework Phases' condensed.

[notes_chain] Processing section 7/8: 'Types of incident response teams'


INFO: HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


[notes_chain] ⚠ JsonOutputParser failed for 'Types of incident response teams' — attempting json_repair.
[notes_chain] ✓ Section 'Types of incident response teams' condensed.

[notes_chain] Processing section 8/8: 'Why is an Incident Response Plan Important'


INFO: HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


[notes_chain] ⚠ JsonOutputParser failed for 'Why is an Incident Response Plan Important' — attempting json_repair.
[notes_chain] ✓ Section 'Why is an Incident Response Plan Important' condensed.
[notes_chain] PDF built: c:\Users\USER\Documents\AI projects\Denseless-local\data\notes\student_1019_Incident_Response_20260624_103625.pdf

[notes_chain] ═══ Notes generation complete ═══
[notes_chain] PDF: c:\Users\USER\Documents\AI projects\Denseless-local\data\notes\student_1019_Incident_Response_20260624_103625.pdf
[notes_chain] Sections completed: 8/8
[notes_chain] Total tokens — in: 11,264 | out: 3,543 | total: 14,807
  [token_service] Deducted 14,807 tokens — student: 'student_1019' | chain: run_notes_chain | in: 11,264 | out: 3,543 | remaining: 6,804,798
  [token_guard] ✓ Tokens deducted — student: student_1019 | consumed: 14807 | remaining: 6804798


In [ ]:
# Case: A fast student

# Enter student_id as student_{number used for vector store}
pdf_path = run_notes_chain(
      student_id    = "student_1019",
      topic_map     = chunks_topic,
      current_topic = "Interconnection Structures",
      weak_topics=[],
      strong_topics = [],
      course="CSC 315",
      learning_pace = "fast",
      llm           = llm,
      embedder      = embedder,
      store         = store,
  )

INFO:google_genai.models:AFC is enabled with max remote calls: 10.


  [token_service] Tokens remaining for 'student_1019': 9,848,255
  [token_guard] Checking tokens — student: student_1019 | remaining: 9848255 | chain: run_notes_chain

[notes_chain] ═══ Starting notes generation ═══
[notes_chain] Mode: Gemini API
[notes_chain] Student: student_1019 | Topic: Interconnection Structures | Pace: fast
[notes_chain] Sections to process: 15

[notes_chain] Processing section 1/15: 'COMPUTER ARCHITECTURE & ORGANIZATION LECTURE (WEEK'


INFO:httpx:HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash-lite:generateContent "HTTP/1.1 200 OK"


[notes_chain] ✓ Section 'COMPUTER ARCHITECTURE & ORGANIZATION LECTURE (WEEK' condensed.
[notes_chain] ⏳ Throttle: sleeping 5s before next section.


INFO:google_genai.models:AFC is enabled with max remote calls: 10.


...

[notes_chain] ✓ Section 'Word length' condensed.
[notes_chain] ⏳ Throttle: sleeping 5s before next section.


INFO:google_genai.models:AFC is enabled with max remote calls: 10.



[notes_chain] Processing section 15/15: 'INSTRUCTION SEQUENCING'


INFO:httpx:HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash-lite:generateContent "HTTP/1.1 200 OK"


[notes_chain] ✓ Section 'INSTRUCTION SEQUENCING' condensed.
[notes_chain] PDF built: c:\Users\USER\Documents\AI projects\DenseLess\data\notes\student_1019_Interconnection_Structures_20260428_224202.pdf

[notes_chain] ═══ Notes generation complete ═══
[notes_chain] PDF: c:\Users\USER\Documents\AI projects\DenseLess\data\notes\student_1019_Interconnection_Structures_20260428_224202.pdf
[notes_chain] Sections completed: 15/15
[notes_chain] Total tokens — in: 15,872 | out: 3,427 | total: 19,299
  [token_service] Deducted 19,299 tokens — student: 'student_1019' | chain: run_notes_chain | in: 15,872 | out: 3,427 | remaining: 9,828,956
  [token_guard] ✓ Tokens deducted — student: student_1019 | consumed: 19299 | remaining: 9828956


### QA chain

In [17]:
from app.agent.rag.chains.qa_chain import run_qa_chain, list_conversations, view_ltm
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_ollama.chat_models import ChatOllama
from dotenv import load_dotenv
import json

In [18]:
# Instantiate your llm (local or paid)

llm = ChatOllama(model="mistral:latest")
# load_dotenv()
# api_key = os.environ.get('GOOGLE_API_KEY')

# llm = ChatGoogleGenerativeAI(
#     model="gemini-2.5-flash-lite", 
#     project="denseless",       
#     location="us-central1",              
#     vertexai=True,
# )

In [19]:
# Check out the list of available convos created in the past
print(list_conversations("student_1019"))

[qa_chain] Found 5 conversation(s) for student 'student_1019'.
['COA_chat_1', 'PQ_chat_1', 'PQ_chat_2', 'PQ_chat_3', 'SE_chat_1']


In [ ]:
# Pass the parameters for qa_chain

# Enter student_id as student_{number used for vector store}

response = run_qa_chain(
    "student_1001",
    "Please don't give me analogies, I just want the raw technical explanation. I hate analogies in any explanaton I ask for. Now, gimme types of security incidents in few sentences that are simple and concise to process",
    "PQ_chat_001",
    False,
    store,
    llm,
    "average",
    "Incident Response.pdf",
    "PQ",
)
print(f"\n\nUser: {response.content['question']} \nModel: {response.content['answer']}")

  [token_service] Tokens remaining for 'student_1001': 994,338
  [token_guard] Checking tokens — student: student_1001 | remaining: 994338 | chain: run_qa_chain
[qa_chain] Starting QA — student: 'student_1001' | topic: 'Incident Response.pdf' | convo: 'PQ_chat_001' | pace: 'average'
[qa_chain] STM loaded for 'PQ_chat_001' (1 turns summarised).
[qa_chain] STM loaded for 'PQ_chat_001' (1 turns summarised).
[qa_chain] STM context loaded (657 chars).
[qa_chain] No LTM file found — starting fresh for student 'student_1001'.
[qa_chain] LTM: store is empty for student 'student_1001'.


INFO: HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


[qa_chain] Question reformulated.
[Retriever] Phase 1 — Semantic search (top_k=5, threshold=0.4): 'Provide a straightforward, non-analogical summary of common '
Querying Chroma (top_k=5, threshold=0.4): 'Represent this sentence for searching relevant passages: Pro...'
  → Retrieved 5 chunk(s), 5 passed threshold.
[Retriever] → 5 seed chunk(s) passed threshold.
[Retriever] Phase 2 — Section context expansion...
[Retriever] Expanding context across 1 section(s): {'Types of security incidents'}
[Retriever]   → Section 'Types of security incidents': 14 chunk(s) fetched.
[Retriever] → Expansion complete: 5 seed → 14 unique chunk(s) after deduplication.
[Retriever] → Final result: 14 chunk(s) returned to chain.
[qa_chain] Retrieved 14 chunk(s).
1. Types of security incidents
2. In developing incident response strategies, it's important to first understand how security vulnerabilities, threats and incidents relate.
3. A vulnerability is a weakness in the IT or business environment. A threat i

INFO: HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


[qa_chain] QA LLM call succeeded. Grounded: None


INFO: HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


[qa_chain] STM summary updated (1022 chars).
[qa_chain] STM saved for 'PQ_chat_001' (2 turns summarised).


INFO: HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


[qa_chain] LTM raw manager output: []
[qa_chain] LTM: no memories extracted from this exchange.
[qa_chain] QA chain complete for student 'student_1001'.
  [token_service] Deducted 1,570 tokens — student: 'student_1001' | chain: run_qa_chain | in: 1,216 | out: 354 | remaining: 992,768
  [token_guard] ✓ Tokens deducted — student: student_1001 | consumed: 1570 | remaining: 992768


User: Please don't give me analogies, I just want the raw technical explanation. I hate analogies in any explanaton I ask for. Now, gimme types of security incidents in few sentences that are simple and concise to process 
Model: Common security incidents include:  1. Unauthorized access attempts to systems
or data. 2. Privilege escalation attacks, where an attacker gains higher-level
access than initially granted. 3. Insider threats, malicious actions by
individuals within the organization. 4. Phishing attacks, attempting to obtain
sensitive information through deception. 5. Malware attacks, using harmful
soft

: 

In [16]:
# Obsereve the LLM's ltm about student
view_ltm("student_1001")

'No long-term memories stored yet for student.'

### Quizz chain

In [17]:
from app.services.quiz_router import handle_quiz_request
import os 
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI

In [ ]:
# Instantiate your llm (local or paid)

llm = ChatOllama(model="mistral:latest")
# load_dotenv()
# api_key = os.environ.get('GOOGLE_API_KEY')

# llm = ChatGoogleGenerativeAI(
#     model="gemini-2.5-flash-lite", 
#     project="denseless",       
#     location="us-central1",              
#     vertexai=True,
# )

INFO: The user provided project/location will take precedence over the Vertex AI API key from the environment variable.


In [23]:
# It supports time-travel testing (simulating a future date to test scheduled quizzes)
# Use the function below to interact with the quiz chain; The date must
# be in the format YYYY-MM-DD

# Enter student_id as student_{number used for vector store}

result = handle_quiz_request(
    student_id      = "student_1001",
    course          = "PQ",
    topic           = "Incident Response",
    store           = store,
    llm             = llm,
    simulated_today = "2026-07-12"  # Optional: ISO date string
)
if result["status"] == "ok":
    quiz = result["quiz"]       # dict — 10 questions
else:
    print(result["message"])    # e.g. "Read your notes first."

INFO: AFC is enabled with max remote calls: 10.


[quiz_router] Request — student: 'student_1001', course: 'PQ', topic: 'Incident Response'
[quiz_router] 🕒 DEV MODE: Simulating today as 2026-07-12
[quiz_router] New topic initialised in profile: 'Incident Response'
[quiz_router] Phase detected: 'pre_test'
[quiz_router] Retrieval strategy — broad
  [token_service] Tokens remaining for 'student_1001': 998,344
  [token_guard] Checking tokens — student: student_1001 | remaining: 998344 | chain: run_quiz_chain
[quiz_chain] Starting — student: 'student_1001', course: 'PQ', topic: 'Incident Response'
[quiz_chain] Broad retrieval — no weak areas on record.
[Retriever] Topic sweep — fetching all chunks from store...
[Retriever] → 43 chunk(s) merged into existing sections (duplicate titles collapsed).
[Retriever] → 51 chunk(s) across 8 section(s).
[Retriever] → Topic map built: 8 section(s), all chunks retained.
[quiz_chain] Retrieved 8 chunk(s) for topic 'Incident Response'.
[quiz_chain] Prompt built — chunks: 8, source pages: [1, 2, 3, 4, 5]
[

INFO: HTTP Request: POST https://us-central1-aiplatform.googleapis.com/v1beta1/projects/denseless/locations/us-central1/publishers/google/models/gemini-2.5-flash-lite:generateContent "HTTP/1.1 200 OK"


[quiz_chain] JsonOutputParser succeeded.
[quiz_chain] Running per-question source mapping...
[Retriever] Phase 1 — Semantic search (top_k=3, threshold=0.4): 'What are two examples of common security incidents that orga'
Querying Chroma (top_k=3, threshold=0.4): 'Represent this sentence for searching relevant passages: Wha...'
  → Retrieved 3 chunk(s), 3 passed threshold.
[Retriever] → 3 seed chunk(s) passed threshold.
[Retriever] Phase 2 — Section context expansion...
[Retriever] Expanding context across 1 section(s): {'Types of security incidents'}
[Retriever]   → Section 'Types of security incidents': 14 chunk(s) fetched.
[Retriever] → Expansion complete: 3 seed → 14 unique chunk(s) after deduplication.
[Retriever] → Final result: 14 chunk(s) returned to chain.
[quiz_chain] Q1 mapped — pages: [1, 2], section: Types of security incidents
[Retriever] Phase 1 — Semantic search (top_k=3, threshold=0.4): 'What is the primary function of an incident response plan? A'
Querying Chroma (top_k

### Eval chain

In [5]:
from app.agent.rag.chains.eval_chain import run_eval_chain
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_ollama import ChatOllama

In [10]:
# Instantiate your llm (local or paid)

llm = ChatOllama(model="mistral:latest")
# load_dotenv()
# api_key = os.environ.get('GOOGLE_API_KEY')

# llm = ChatGoogleGenerativeAI(
#     model="gemini-2.5-flash-lite", 
#     project="denseless",       
#     location="us-central1",              
#     vertexai=True,
# )

In [ ]:
# Interact with the eval chain using the function below
# quiz_phase = pre_test | post_test | revision
# pre_test and post_test are for measuring comprehension
# revision is for retention over a 2wk period - However, you
# can use the simulator date to bypass the wait
# date is in the format YYYY-MM-DD

# Enter student_id as student_{number used for vector store}

response = run_eval_chain(
    student_id="student_1019",
    topic="Incident Response",
    quiz_phase="revision",
    quiz_path="../../data/quizzes/student_1019_incident_response_20260706_122106.json",
    llm=llm,
    simulated_date="2026-07-12"
)
print(response.content["questions"][0]["score"])
print(response.content["questions"][0]["explanation"])

  [token_service] Tokens remaining for 'student_1019': 6,703,563
  [token_guard] Checking tokens — student: student_1019 | remaining: 6703563 | chain: run_eval_chain
[eval_chain] Quiz loaded from ..\..\data\quizzes\student_1019_incident_response_20260706_122106.json
[eval_chain] Starting eval — student: 'student_1019' | topic: 'Incident Response' | phase: 'revision' | questions: 10
[eval_chain] Grading question 1/10...


INFO:httpx:HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


[eval_chain] Question graded. Score: 0.5
[eval_chain] Grading question 2/10...


INFO:httpx:HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


[eval_chain] Question graded. Score: 1.0
[eval_chain] Grading question 3/10...


...

[eval_chain] Question graded. Score: 0.5
[eval_chain] Grading question 10/10...


INFO:httpx:HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


[eval_chain] Question graded. Score: 1.0
[eval_chain] All 10 questions graded.
[eval_chain] Total score: 8.5 / 10
[eval_chain] Pre-resolution — strong: {'Types of security incidents', 'Why is an Incident Response Plan Important', 'Incident response frameworks: Phases of incident response', 'How to create an incident response plan', 'Introduction'} | weak: {'Incident response frameworks: Phases of incident response', 'Types of security incidents', 'Introduction'}
[eval_chain] Post-resolution — strong: {'How to create an incident response plan', 'Why is an Incident Response Plan Important'} | weak: {'Incident response frameworks: Phases of incident response', 'Types of security incidents', 'Introduction'}


INFO:httpx:HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


[eval_chain] Session feedback generated.
[eval_chain] Section classifications updated for topic 'Incident Response'.
[eval_chain] Revision score → retention[Incident Response] (attempt 2).
[eval_chain] Revision date entry for 2026-07-12 marked completed.
[eval_chain] Graded quiz saved to ..\..\data\quizzes\student_1019_incident_response_20260706_122106.json
[eval_chain] Profile saved for student 'student_1019'.
[eval_chain] Eval chain complete for student 'student_1019'.
  [token_service] Deducted 6,281 tokens — student: 'student_1019' | chain: run_eval_chain | in: 5,698 | out: 583 | remaining: 6,697,282
  [token_guard] ✓ Tokens deducted — student: student_1019 | consumed: 6281 | remaining: 6697282
0.5
Partially correct — you listed three phases: Containment, Detection, and Eradication.
